# 🎓 Lab Tutorial: Controlling Nodes & Relationships in GraphRAG (LlamaIndex + Ollama)

In `graphrag_student_tutorial.ipynb`, we used `SimpleLLMPathExtractor`, which lets the LLM freely decide what entities and relationships to extract from the documents. While convenient, free-form extraction often produces inconsistent labels (e.g., `CO_FOUNDED`, `FOUNDED_BY`, `CREATED` for the same concept) or noisy nodes.

In this tutorial, you will learn **two ways to take full control** over your Property Graph:

1. **Approach 1 — Schema-Guided Extraction (`SchemaLLMPathExtractor`)**: You define the allowed **entity types**, **relationship types**, and **valid `(Subject, Relation, Object)` schema rules**. The LLM still reads your documents automatically, but is strictly constrained to your ontology.
2. **Approach 2 — Manual / Exact Graph Construction (`EntityNode` & `Relation`)**: You explicitly define every single node instance, property, and edge in Python and upsert them directly into `SimplePropertyGraphStore` with zero LLM guesswork.

---

## 1. Setup & Environment Configuration
Just like in the first tutorial, we apply `nest_asyncio` for Jupyter compatibility, load the 4 sample documents from `data/`, and configure local **Ollama** models.

> **Tip:** For schema-constrained extraction, setting `temperature=0.0` (instead of `0.9`) helps local LLMs follow the JSON schema much more reliably.

In [14]:
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Patch the running asyncio event loop so LlamaIndex can run smoothly.
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# Configure Local Ollama LLM (temperature=0.0 for deterministic schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)

# Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# Load the 4 documents from data/
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

Loaded 4 documents from /home/ashok/Documents/graph_rag/data


--- 
## 2. Approach 1: Schema-Guided Extraction (`SchemaLLMPathExtractor`)

Instead of `SimpleLLMPathExtractor`, we use **`SchemaLLMPathExtractor`**, which accepts three schema controls:

1. **`possible_entities`**: A `typing.Literal[...]` listing the exact node types allowed (e.g., `PERSON`, `COMPANY`, `INITIATIVE`, `TECHNOLOGY`, `LOCATION`).
2. **`possible_relations`**: A `typing.Literal[...]` listing the exact edge labels allowed (e.g., `CO_FOUNDED`, `INVESTED_IN`, `ADVISES`, `LICENSES_TO`).
3. **`kg_validation_schema`**: A list of allowed `(SubjectType, Relation, ObjectType)` triples (or a dict mapping each entity type to its allowed outgoing relations).
4. **`strict=True`**: Drops any extracted triplet that does not conform to your schema. *(Set `strict=False` if you want your schema to act as a soft hint rather than a hard filter.)*

In [6]:
from typing import Literal
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor

# 1. Define allowed Node Types (Entities)
entities = Literal[
    "PERSON",
    "COMPANY",
    "INITIATIVE",
    "TECHNOLOGY",
    "LOCATION",
    "REGULATOR",
]

# 2. Define allowed Relationship Types (Edges)
relations = Literal[
    "CO_FOUNDED",
    "WORKS_AT",
    "ADVISES",
    "SERVES_ON_BOARD_OF",
    "COLLABORATED_WITH",
    "INVESTED_IN",
    "FUNDED",
    "JOINT_VENTURE_WITH",
    "LICENSES_TO",
    "DEVELOPS",
    "TESTBED_FOR",
    "HEADQUARTERED_IN",
    "INVESTIGATES",
]

# 3. Define valid (SourceEntityType, Relation, TargetEntityType) rules
validation_schema = [
    ("PERSON", "CO_FOUNDED", "COMPANY"),
    ("PERSON", "WORKS_AT", "COMPANY"),
    ("PERSON", "ADVISES", "COMPANY"),
    ("PERSON", "SERVES_ON_BOARD_OF", "COMPANY"),
    ("PERSON", "COLLABORATED_WITH", "PERSON"),
    ("PERSON", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "INVESTED_IN", "COMPANY"),
    ("COMPANY", "FUNDED", "INITIATIVE"),
    ("COMPANY", "JOINT_VENTURE_WITH", "COMPANY"),
    ("COMPANY", "LICENSES_TO", "COMPANY"),
    ("COMPANY", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "TESTBED_FOR", "INITIATIVE"),
    ("COMPANY", "HEADQUARTERED_IN", "LOCATION"),
    ("REGULATOR", "INVESTIGATES", "PERSON"),
    ("REGULATOR", "INVESTIGATES", "COMPANY"),
]

# 4. Create the Schema-Constrained Extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True,              # Enforce schema strictly (set False for soft guidance)
    max_triplets_per_chunk=10,
    num_workers=1,
)

print("Constructing Schema-Guided Property Graph Index via Ollama...")
index_schema = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    use_async=False,
    show_progress=True,
)
print("Schema-guided graph index built successfully!")

Constructing Schema-Guided Property Graph Index via Ollama...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-25 14:40:56,365 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-25 14:41:29,318 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-25 14:42:00,287 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-25 14:42:35,729 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-25 14:43:08,633 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-11-25 14:43:10,730 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/54 [00:00<?, ?it/s]

2026-11-25 14:43:12,652 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:43:14,553 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:43:16,462 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:43:18,374 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:43:20,288 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:43:21,059 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Schema-guided graph index built successfully!


### 2.1 Inspect the Extracted Triplets
Let's print out the extracted `(Subject [Label]) --[Relation]--> (Object [Label])` triples from the graph store to verify that every node and edge adheres to our schema.

In [7]:
from llama_index.core.graph_stores.types import EntityNode

# Note: SimplePropertyGraphStore.get_triplets() returns [] if called with no filters.
# Access `.graph.get_triplets()` directly to inspect all triplets in the store.
all_triplets = index_schema.property_graph_store.graph.get_triplets()

# Filter to Entity-to-Entity triplets (excluding document ChunkNode MENTIONS edges)
entity_triplets = [
    (subj, rel, obj)
    for subj, rel, obj in all_triplets
    if isinstance(subj, EntityNode) and isinstance(obj, EntityNode)
]

print(f"Total extracted entity triplets: {len(entity_triplets)} (out of {len(all_triplets)} total graph edges)\n")

for subj, rel, obj in entity_triplets:
    print(f"({subj.name} [{subj.label}]) --[{rel.label}]--> ({obj.name} [{obj.label}])")

Total extracted entity triplets: 17 (out of 17 total graph edges)

(Dr. Elena Vance [PERSON]) --[WORKS_AT]--> (Aurora Technologies [COMPANY])
(Synapse Dynamics [COMPANY]) --[INVESTED_IN]--> (NeuraCore Fab [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[HEADQUARTERED_IN]--> (Seattle, Washington [LOCATION])
(Synapse Dynamics [COMPANY]) --[DEVELOPS]--> (Synapse-OS cognitive architecture [TECHNOLOGY])
(Aurora Technologies Inc. [COMPANY]) --[JOINT_VENTURE_WITH]--> (Helios Capital [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[JOINT_VENTURE_WITH]--> (Aether Labs [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[LICENSES_TO]--> (Synapse Dynamics [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[LICENSES_TO]--> (Helios Capital [COMPANY])
(Helios Capital [COMPANY]) --[INVESTED_IN]--> (Aurora Technologies [COMPANY])
(Synapse Dynamics [COMPANY]) --[DEVELOPS]--> (Agile biomimetic humanoid robots [TECHNOLOGY])
(Helios Capital [COMPANY]) --[INVESTED_IN]--> (Aether Labs [COMPANY])
(Aurora Te

### 2.2 Visualize the Schema-Guided Graph

In [8]:
from IPython.display import IFrame

index_schema.property_graph_store.save_networkx_graph(name="./graph_schema.html")
IFrame(src="./graph_schema.html", width="100%", height="700px")

### 2.3 Query the Schema-Guided Graph

In [9]:
query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

schema_query_engine = index_schema.as_query_engine(include_text=True, similarity_top_k=3)
response_schema = schema_query_engine.query(query_text)

print("=== 💡 SCHEMA-GUIDED GRAPHRAG RESPONSE ===")
print(response_schema.response)

2026-11-25 14:44:43,880 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-25 14:44:43,902 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:45:17,836 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 SCHEMA-GUIDED GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics and Project Nexus through Helios Capital in several ways:

1. **Intellectual Property and Licensing**: Dr. Elena Vance, as a co-founder and former Chief Scientist of Aurora Technologies, brought the "CryoQubit" patent portfolio to the table. Aurora Technologies licenses optical communication protocols to Synapse Dynamics, an arrangement that Dr. Vance recused herself from due to undisclosed personal ties. This connection is further strengthened by the fact that Synapse Dynamics develops agile biomimetic humanoid robots, which could benefit from the quantum compression technologies developed by Aurora Technologies.

2. **Project Nexus**: Helios Capital, which has invested in both Aurora Technologies and Synapse Dynamics, played a crucial role in stitching these entities together into Project Nexus. Helios Capital's involvement facilitated the joint venture between Aurora Technologies and Synapse D

---
## 3. Approach 2: Manually Defining Exact Nodes & Relationships

What if you do **not** want the LLM to extract the graph at all — for example, you already have structured data (from a CSV, SQL database, JSON, or domain experts) and want **100% deterministic control** over every node, property, and edge?

You can build the graph directly using LlamaIndex's `EntityNode`, `Relation`, and `SimplePropertyGraphStore`, and then wrap it with `PropertyGraphIndex.from_existing(...)`.

In [1]:
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Patch the running asyncio event loop so LlamaIndex can run smoothly.
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# Configure Local Ollama LLM (temperature=0.0 for deterministic schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)

# Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# Load the 4 documents from data/
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

Loaded 4 documents from /home/ashok/Documents/graph_rag/data


In [2]:
from llama_index.core.graph_stores import (
    SimplePropertyGraphStore,
    EntityNode,
    Relation,
)

# 1. Create an empty in-memory Property Graph Store
manual_store = SimplePropertyGraphStore()

# 2. Explicitly define your Nodes (with labels and rich properties)
elena = EntityNode(
    name="Dr. Elena Vance",
    label="PERSON",
    properties={"role": "Co-Founder & Chief Scientific Advisor", "specialty": "CryoQubit & Quantum Compression"},
)
marcus = EntityNode(
    name="Marcus Thorne",
    label="PERSON",
    properties={"role": "Co-Founder & CEO"},
)
kenji = EntityNode(
    name="Dr. Kenji Sato",
    label="PERSON",
    properties={"role": "Founder & CEO", "background": "Former MIT robotics faculty"},
)
julian = EntityNode(
    name="Julian Mercer",
    label="PERSON",
    properties={"role": "Principal Investment Director"},
)
aurora = EntityNode(
    name="Aurora Technologies",
    label="COMPANY",
    properties={"headquarters": "Seattle, WA", "founded": 2021, "focus": "Quantum computing hardware"},
)
helios = EntityNode(
    name="Helios Capital",
    label="COMPANY",
    properties={"headquarters": "New York City", "fund": "Frontier Tech Fund IV ($1.2B)"},
)
aether = EntityNode(
    name="Aether Labs",
    label="COMPANY",
    properties={"type": "Joint Venture", "established": "September 2023"},
)
synapse = EntityNode(
    name="Synapse Dynamics",
    label="COMPANY",
    properties={"headquarters": "Boston", "founded": 2019, "focus": "Biomimetic humanoid robots"},
)
nexus = EntityNode(
    name="Project Nexus",
    label="INITIATIVE",
    properties={"status": "Stealth consortium", "goal": "Quantum-powered autonomous robotics"},
)

nodes = [elena, marcus, kenji, julian, aurora, helios, aether, synapse, nexus]

# 3. Explicitly define your Relationships between Nodes
relations = [
    Relation(label="CO_FOUNDED", source_id=elena.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="CO_FOUNDED", source_id=marcus.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="ADVISES", source_id=elena.id, target_id=synapse.id, properties={"since": "March 2024", "title": "Chief Scientific Advisor"}),
    Relation(label="COLLABORATED_WITH", source_id=elena.id, target_id=kenji.id, properties={"location": "Stanford University", "year": 2016}),
    Relation(label="CO_FOUNDED", source_id=kenji.id, target_id=synapse.id, properties={"year": 2019}),
    Relation(label="WORKS_AT", source_id=julian.id, target_id=helios.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aurora.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aether.id),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aurora.id, properties={"amount": "$30M", "equity": "15%"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=synapse.id, properties={"amount": "$60M", "equity": "22%"}),
    Relation(label="JOINT_VENTURE_WITH", source_id=aurora.id, target_id=aether.id, properties={"partner": "Helios Capital"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aether.id, properties={"amount": "$45M"}),
    Relation(label="LICENSES_TO", source_id=aurora.id, target_id=synapse.id, properties={"technology": "Optical Communication Protocols"}),
    Relation(label="FUNDED", source_id=helios.id, target_id=nexus.id, properties={"mode": "Covert"}),
    Relation(label="TESTBED_FOR", source_id=synapse.id, target_id=nexus.id),
]

# 4. Upsert Nodes FIRST, then Relations into the Graph Store
manual_store.upsert_nodes(nodes)
manual_store.upsert_relations(relations)

# 5. Wrap the populated store in a PropertyGraphIndex (no LLM extraction needed!)
# Note: use_async=False avoids nested-event-loop / sniffio errors in Jupyter
index_manual = PropertyGraphIndex.from_existing(
    property_graph_store=manual_store,
    llm=llm,
    embed_model=embed_model,
    use_async=False,
)

print(f"Manual graph created with {len(nodes)} nodes and {len(relations)} relationships!")

Manual graph created with 9 nodes and 15 relationships!


### 3.1 Visualize & Query the Manually Built Graph

In [6]:
from IPython.display import IFrame

index_manual.property_graph_store.save_networkx_graph(name="./graph_schema.html")
IFrame(src="./graph_schema.html", width="100%", height="700px")

In [7]:
# Export and render the manual graph
index_manual.property_graph_store.save_networkx_graph(name="./graph_manual.html")
IFrame(src="./graph_manual.html", width="100%", height="650px")

In [8]:
# Query the manually constructed graph
# Note: use_async=False ensures synchronous retrieval (avoids sniffio/asyncio errors in Jupyter)
manual_query_engine = index_manual.as_query_engine(include_text=False, use_async=False)
response_manual = manual_query_engine.query(
    "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"
)

print("=== 💡 MANUAL GRAPHRAG RESPONSE ===")
print(response_manual.response)

2026-11-25 14:58:20,629 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-25 14:58:27,799 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-25 14:58:28,323 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-25 14:58:37,812 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 MANUAL GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics through her collaboration with Dr. Kenji Sato, who co-founded Synapse Dynamics. Additionally, Synapse Dynamics serves as a testbed for Project Nexus. Helios Capital, which invested in both Synapse Dynamics and Synapse Dynamics' Project Nexus, also invested in Aurora Technologies, where Dr. Elena Vance co-founded and advises Synapse Dynamics. This connection links Dr. Elena Vance to Synapse Dynamics and Project Nexus through Helios Capital's investments.


---
## 4. Quick Comparison Summary

| Method | Who Decides Node/Edge Types? | Who Decides Exact Instances? | Best Use Case |
| :--- | :--- | :--- | :--- |
| **`SimpleLLMPathExtractor`** | LLM | LLM | Quick exploration of unfamiliar text |
| **`SchemaLLMPathExtractor` (`strict=True`)** | **You** (`Literal` + `kg_validation_schema`) | LLM (constrained to your schema) | Production RAG over unstructured documents with a clean ontology |
| **`SchemaLLMPathExtractor` (`strict=False`)** | **You** (as a soft guide) + LLM | LLM | Semi-structured domains where new types may appear |
| **Manual `EntityNode` + `Relation`** | **You** | **You** | Structured data (CSV/DB/APIs) or curated gold-standard knowledge graphs |

In [ ]:
###############